# Phase 4 — Tri-Modal Cross-Attention Fusion

**Thesis:** Tri-Modal Depression Risk Detection  
**Author:** Md. Mursalin, Netrokona University  

## What happens in this notebook

1. Load pre-processed tri-modal features (face AU, audio MFCC, text TF-IDF)
2. Train **TriModalFusionModel** with cross-modal attention + fairness loss
3. Visualize attention weights — *which modality attended to which*
4. Confusion matrix + ROC curve
5. Fairness report: F1 by gender group

### Why Cross-Modal Attention?

> Simple concatenation assumes all modalities contribute equally and independently.
> Cross-modal attention lets the model learn *when face is more informative than audio*
> and vice versa — dynamically, per sample. This is the key novel contribution.

In [ ]:
!git clone https://github.com/DevMursLab/Thesis.git depression_thesis
%cd depression_thesis
!pip install -q -r requirements.txt

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Load Pre-Processed Features
Run all three preprocessing scripts first (or mount Google Drive with processed .npy files).

In [ ]:
# Mount DAIC-WOZ from Drive and run preprocessing
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/daicwoz depression_thesis/daicwoz

!python src/preprocessing/daic_preprocess.py
!python src/preprocessing/audio_preprocess.py
!python src/preprocessing/text_preprocess.py

In [ ]:
import numpy as np

Xf = np.load('data/processed/daic_faces/X_train.npy')
Xa = np.load('data/processed/daic_audio/X_train.npy')
Xt = np.load('data/processed/daic_text/X_train.npy')
y  = np.load('data/processed/daic_faces/y_train.npy')

print(f'Face  features: {Xf.shape}')   # (N, 200, 20)
print(f'Audio features: {Xa.shape}')   # (N, 300, 120)
print(f'Text  features: {Xt.shape}')   # (N, 10000)
print(f'Labels: {y.shape} | Dep: {y.sum()} ({100*y.mean():.1f}%)')

## 2. Train Tri-Modal Fusion Model

In [ ]:
!python src/training/train_fusion.py

## 3. Attention Weight Visualization

In [ ]:
import sys, torch, numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '.')
from src.models.fusion_attention import TriModalFusionModel
from configs.config import N_AU_FEATURES, N_MFCC, VOCAB_SIZE

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = TriModalFusionModel(n_au=N_AU_FEATURES, n_mfcc=N_MFCC*3,
                              vocab_size=VOCAB_SIZE, embed_dim=256).to(device)
model.load_state_dict(torch.load('results/fusion_trimodal_best.pth', map_location=device))
model.eval()

Xf_dv = np.load('data/processed/daic_faces/X_dev.npy')
Xa_dv = np.load('data/processed/daic_audio/X_dev.npy')
Xt_dv = np.load('data/processed/daic_text/X_dev.npy')
y_dv  = np.load('data/processed/daic_faces/y_dev.npy')

with torch.no_grad():
    logits, attn = model(
        torch.tensor(Xf_dv).to(device),
        torch.tensor(Xa_dv).to(device),
        torch.tensor(Xt_dv).to(device),
        return_attention=True
    )

# Average attention weights across dev set
fa_audio = attn['face_attends_to']['audio'].cpu().numpy().mean()
fa_text  = attn['face_attends_to']['text'].cpu().numpy().mean()
aa_face  = attn['audio_attends_to']['face'].cpu().numpy().mean()
aa_text  = attn['audio_attends_to']['text'].cpu().numpy().mean()
ta_face  = attn['text_attends_to']['face'].cpu().numpy().mean()
ta_audio = attn['text_attends_to']['audio'].cpu().numpy().mean()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
modalities = ['Audio', 'Text']

for ax, title, vals in zip(axes,
    ['Face attends to', 'Audio attends to', 'Text attends to'],
    [[('Audio', fa_audio), ('Text', fa_text)],
     [('Face', aa_face),  ('Text', aa_text)],
     [('Face', ta_face),  ('Audio', ta_audio)]]):
    labels = [v[0] for v in vals]
    values = [v[1] for v in vals]
    bars = ax.bar(labels, values, color=['steelblue','tomato','seagreen'][:len(labels)])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Mean Attention Weight')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=10)

plt.suptitle('Cross-Modal Attention Weights (Dev Set Average)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/phase4_attention_weights.png', dpi=150)
plt.show()
print('Higher weight = this modality was more attended to')

## 4. Evaluation — Confusion Matrix + ROC + Fairness

In [ ]:
import json
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc as sk_auc)
from torch.utils.data import TensorDataset, DataLoader

g_dv = np.load('data/processed/daic_faces/gender_dev.npy')

dev_dl = DataLoader(
    TensorDataset(torch.tensor(Xf_dv), torch.tensor(Xa_dv),
                  torch.tensor(Xt_dv), torch.tensor(y_dv), torch.tensor(g_dv)),
    batch_size=32)

all_p, all_pr, all_t, all_g = [], [], [], []
model.eval()
with torch.no_grad():
    for xf, xa, xt, yb, gb in dev_dl:
        logits = model(xf.to(device), xa.to(device), xt.to(device))
        all_pr.extend(torch.softmax(logits,1)[:,1].cpu().numpy())
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_t.extend(yb.numpy()); all_g.extend(gb.numpy())

t=np.array(all_t); p=np.array(all_p); pr=np.array(all_pr); g=np.array(all_g)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(t, p)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0], cmap='Blues',
            xticklabels=['Not Dep','Depressed'],
            yticklabels=['Not Dep','Depressed'])
axes[0].set_title('Confusion Matrix — Tri-Modal Fusion'); axes[0].set_ylabel('True')

fpr, tpr, _ = roc_curve(t, pr)
roc_auc = sk_auc(fpr, tpr)
axes[1].plot(fpr, tpr, lw=2, color='darkorange', label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curve — Tri-Modal Fusion'); axes[1].legend()

plt.tight_layout()
plt.savefig('results/figures/phase4_confusion_roc.png', dpi=150)
plt.show()

print(classification_report(t, p, target_names=['Not Depressed','Depressed'], zero_division=0))

from sklearn.metrics import f1_score
print('=== FAIRNESS REPORT ===')
print(f'F1 (male):   {f1_score(t[g==0],p[g==0],zero_division=0):.4f}')
print(f'F1 (female): {f1_score(t[g==1],p[g==1],zero_division=0):.4f}')
print(f'Gap:         {abs(f1_score(t[g==0],p[g==0],zero_division=0)-f1_score(t[g==1],p[g==1],zero_division=0)):.4f}')
print('(Gap < 0.1 = fair model)')

with open('results/metrics/phase4_metrics.json') as f:
    print('\n', json.dumps(json.load(f), indent=2))